# 02 — Leakage-safe preprocessing

**Purpose.** Turn the raw Excel file into model-ready train/test tables, using *only* information from the training rows to learn anything (imputation models, scaling, label mapping).

**Reads:** `../data/raw/dataset.xlsx` (Harvard Dataverse, DOI 10.7910/DVN/VM4OR3)
**Writes:** `../data/processed/` — `X_raw_*.csv` (14 features), `X_ext_*.csv` (19 features), `y_*.csv`, `label_mapping.json`

**Built-in checks:** the dataset shape is verified on load; MICE convergence is checked (warning captured, `n_iter_` printed); and a final cell re-runs the shared `src/preprocessing/cv_pipeline.py::preprocess_fold` on the same split and asserts that it reproduces this notebook's output exactly, so this notebook and the cross-validation code cannot drift apart.


## 0. Setup

**What this does:** imports libraries, makes the project's `src/` folder importable (needed for the cross-check in section 11), prints library versions, and confirms the raw dataset file is where we expect it.

**Why print versions:** results depend on library versions (e.g. `IterativeImputer` behaviour). Recording them makes the run reproducible and lets you compare with `results/environment.txt`.

In [1]:
import sys, os, json, warnings
sys.path.append('..')   # so `src.preprocessing.cv_pipeline` can be imported

import numpy as np
import pandas as pd
import sklearn

print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)

RAW_PATH = "../data/raw/dataset.xlsx"
if not os.path.exists(RAW_PATH):
    raise FileNotFoundError(
        f"{RAW_PATH} not found. Download the dataset from Harvard Dataverse "
        "(DOI 10.7910/DVN/VM4OR3), rename the file to dataset.xlsx and place it in data/raw/ "
        "(data/ is git-ignored, so it is not in the repo)."
    )
print("Found raw dataset:", RAW_PATH)

Python      : 3.13.9
pandas      : 3.0.5
numpy       : 2.5.2
scikit-learn: 1.9.0
Found raw dataset: ../data/raw/dataset.xlsx


## 1. Load the raw data

**What this does:** reads the Excel file into a DataFrame and prints its shape, column names, and first rows.

**Expected:** `(12085, 15)` — 12,085 patients, 14 feature columns + the `Disease` target. The assertion stops the notebook if that is not true, because every later number in the project assumes this exact dataset.

In [2]:
df = pd.read_excel(RAW_PATH)
print("shape:", df.shape)
assert df.shape == (12085, 15), f"Unexpected shape {df.shape} - wrong file/version?"
print(list(df.columns))
df.head()

shape: (12085, 15)
['Age', 'Gender', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4', 'Disease']


,Age,Gender,ESR,CRP,RF,Anti-CCP,HLA-B27,ANA,Anti-Ro,Anti-La,Anti-dsDNA,Anti-Sm,C3,C4,Disease
0,70,Male,39.0,18.6,34.2,29.9,Positive,Negative,Positive,Negative,Positive,Positive,NaN,27.0,Rheumatoid Arthritis
1,39,Female,26.0,21.7,35.5,28.9,Negative,NaN,Positive,NaN,Positive,NaN,100.0,66.0,Rheumatoid Arthritis
2,36,Female,41.0,15.6,21.3,21.3,Negative,Negative,NaN,Positive,Negative,NaN,158.0,12.0,Rheumatoid Arthritis
3,35,Male,43.0,23.4,26.0,39.0,NaN,NaN,Positive,Positive,NaN,NaN,119.0,41.0,Rheumatoid Arthritis
4,37,Female,30.0,NaN,38.1,30.8,Positive,Negative,Positive,Negative,Positive,Negative,144.0,49.0,Rheumatoid Arthritis


### 1b. Quick sanity look: class balance and missing values

**What this does:** counts patients per disease and missing cells per column.

**Why:** these two facts drive the whole methodology.
* Class imbalance (~5.5 : 1 between the biggest and smallest class) is why we score with **macro-F1** and later handle imbalance explicitly.
* Missing values are why we need imputation, and *where* they are missing decides whether a missingness flag is useful.

**Expected:** 7 classes; Reactive Arthritis is the smallest (~4.3%); `RF` has 1,329 missing values.

In [3]:
counts = df['Disease'].value_counts()
print(counts)
print(f"\nimbalance ratio (largest/smallest): {counts.max()/counts.min():.2f}")
print("\nmissing values per column:")
print(df.isna().sum())

Disease
Rheumatoid Arthritis            2848
Ankylosing Spondylitis          2127
Sjögren's Syndrome              1852
Psoriatic Arthritis             1783
Normal                          1604
Systemic Lupus Erythematosus    1355
Reactive Arthritis               516
Name: count, dtype: int64

imbalance ratio (largest/smallest): 5.52

missing values per column:
Age              0
Gender           0
ESR           1088
CRP           2417
RF            1329
Anti-CCP      3263
HLA-B27       1934
ANA           3746
Anti-Ro       2900
Anti-La       3021
Anti-dsDNA    4713
Anti-Sm       5197
C3            1692
C4            2054
Disease          0
dtype: int64


## 2. Missingness indicators (must be created BEFORE imputation)

**What this does:** adds two 0/1 columns, `RF_was_missing` and `Anti-CCP_was_missing`, recording whether the original value was absent.

**Why:** in the EDA, whether RF is missing was significantly associated with the disease class (p = 0.0055; Anti-CCP borderline, p = 0.0559). That means *missingness itself carries diagnostic signal* (a test may not be ordered for some conditions). Once MICE fills the gap, that information would be lost, so we save it first. Reading these flags is not leakage: they are computed row by row from the row's own data.

In [4]:
df['RF_was_missing'] = df['RF'].isna().astype(int)
df['Anti-CCP_was_missing'] = df['Anti-CCP'].isna().astype(int)

assert df['RF_was_missing'].sum() == df['RF'].isna().sum()
assert df['Anti-CCP_was_missing'].sum() == df['Anti-CCP'].isna().sum()
df[['RF', 'RF_was_missing', 'Anti-CCP', 'Anti-CCP_was_missing']].head(10)

,RF,RF_was_missing,Anti-CCP,Anti-CCP_was_missing
0,34.2,0,29.9,0
1,35.5,0,28.9,0
2,21.3,0,21.3,0
3,26.0,0,39.0,0
4,38.1,0,30.8,0
5,NaN,1,37.3,0
6,22.3,0,NaN,1
7,31.8,0,38.1,0
8,33.4,0,NaN,1
9,37.4,0,25.0,0


## 3. Encode text columns as numbers

**What this does:** maps `Positive/Negative` to `1/0` for the six binary markers and `Male/Female` to `1/0` for Gender. Missing values stay as `NaN`.

**Why:** MICE (next steps) fits regression models, which need numeric input. This mapping is a fixed rule, not something learned from the data, so doing it before the split is safe.

**Check:** every binary column should now contain only `0.0`, `1.0` and `NaN`.

In [5]:
binary_cols = ['HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm']
for col in binary_cols:
    df[col] = df[col].map({'Positive': 1, 'Negative': 0})
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

for col in binary_cols + ['Gender']:
    vals = set(df[col].dropna().unique())
    assert vals <= {0, 1}, f"{col} has unexpected values: {vals}"
print("Binary/gender encoding OK. Remaining NaNs in binary markers:")
print(df[binary_cols].isna().sum())

Binary/gender encoding OK. Remaining NaNs in binary markers:
HLA-B27       1934
ANA           3746
Anti-Ro       2900
Anti-La       3021
Anti-dsDNA    4713
Anti-Sm       5197
dtype: int64


## 4. Train/test split — BEFORE any learned preprocessing

**What this does:** splits patients 80 / 20, **stratified** by disease so both parts keep the same class proportions. The fixed `random_state=42` is reused in every notebook so they all see the identical split.

**Why the order matters:** the imputer and the scaler *learn* parameters from data (regression coefficients, means, standard deviations). If they learned from all rows, information from the test patients would seep into training — **data leakage** — and the final score would look better than real-world performance.

**Important rule for the rest of the project:** the 2,417 test rows are used **once**, in the final evaluation (notebook 10). Model selection uses cross-validation on the training rows only.

We also keep untouched copies (`df_train_raw`, `df_test_raw`) of the split so section 11 can re-run the shared CV code on the same input.

In [6]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df['Disease'], random_state=42
)
df_train = df.loc[train_idx].copy()
df_test  = df.loc[test_idx].copy()

df_train_raw, df_test_raw = df_train.copy(), df_test.copy()   # for the section-11 cross-check

print("train:", df_train.shape, "| test:", df_test.shape)
props = pd.DataFrame({
    'train': df_train['Disease'].value_counts(normalize=True),
    'test':  df_test['Disease'].value_counts(normalize=True),
}).round(3)
props['abs_diff'] = (props['train'] - props['test']).abs()
print(props)
assert props['abs_diff'].max() < 0.005, "Stratification looks off"

train: (9668, 17) | test: (2417, 17)
                              train   test  abs_diff
Disease                                             
Rheumatoid Arthritis          0.236  0.236     0.000
Ankylosing Spondylitis        0.176  0.176     0.000
Sjögren's Syndrome            0.153  0.153     0.000
Psoriatic Arthritis           0.147  0.148     0.001
Normal                        0.133  0.133     0.000
Systemic Lupus Erythematosus  0.112  0.112     0.000
Reactive Arthritis            0.043  0.043     0.000


## 5. MICE imputation (fit on train only)

**What this does:** `IterativeImputer` (scikit-learn's MICE-style imputer) repeatedly predicts each column that has gaps from all the other columns, cycling until the filled-in values stop changing. It is **fit on the training rows only** and merely **applied** (`transform`) to the test rows.

**Parameters**
* `max_iter=50` — a smaller value such as 15 stops before convergence (the imputer converges at about iteration 17) and gives unstable downstream results.
* `random_state=42` — reproducibility.

**Convergence check:** scikit-learn emits a `ConvergenceWarning` if it stops at `max_iter` without converging. We record warnings and print `n_iter_` (the iteration it actually stopped at). We want *no warning* and `n_iter_ < 50`.

**Check:** zero missing values remain in the imputed columns of both sets.

In [7]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.exceptions import ConvergenceWarning

mice_cols = ['ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27', 'ANA',
             'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4']

imputer = IterativeImputer(random_state=42, max_iter=50)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    df_train[mice_cols] = imputer.fit_transform(df_train[mice_cols])   # learn from TRAIN only
n_conv_warn = sum(issubclass(w.category, ConvergenceWarning) for w in caught)
df_test[mice_cols] = imputer.transform(df_test[mice_cols])             # transform only, never fit on test

print("Imputer stopped at iteration:", imputer.n_iter_, "(max 50)")
print("ConvergenceWarnings:", n_conv_warn)
assert df_train[mice_cols].isna().sum().sum() == 0
assert df_test[mice_cols].isna().sum().sum() == 0
print("No missing values remain in the MICE columns.")

Imputer stopped at iteration: 17 (max 50)
ConvergenceWarnings: 0
No missing values remain in the MICE columns.


## 6. Clip imputed values to valid ranges — but do NOT round the binary markers

**What this does**
* Binary markers: `clip(0, 1)` — MICE can overshoot slightly (e.g. 1.03 or -0.02); we pull those back into a valid probability range.
* Continuous labs (ESR, CRP, RF, Anti-CCP, C3, C4): `clip(lower=0)` — negative lab values are impossible.

**Why we do not round the binary markers to a hard 0/1:** for a marker whose true rate in a disease is around 50%, MICE outputs values near 0.5. Rounding pushes them all to one side and *manufactures* an association that is not in the data (example: ANA in Psoriatic Arthritis, true rate ~46.8%, became ~18.6% after rounding). Keeping the continuous estimate improved every model's CV macro-F1.

**What to look at in the output:** the number of *fractional* (not exactly 0 or 1) values per marker — these are the imputed cells we kept as probabilities. The pre-imputation missing count for each marker should be equal or very close to it.

In [8]:
for col in binary_cols:
    df_train[col] = df_train[col].clip(0, 1)
    df_test[col]  = df_test[col].clip(0, 1)

continuous_cols = ['ESR', 'CRP', 'RF', 'Anti-CCP', 'C3', 'C4']
for col in continuous_cols:
    df_train[col] = df_train[col].clip(lower=0)
    df_test[col]  = df_test[col].clip(lower=0)

frac = pd.DataFrame({
    'fractional_in_train': [((df_train[c] != 0) & (df_train[c] != 1)).sum() for c in binary_cols],
    'originally_missing_in_train': [df_train_raw[c].isna().sum() for c in binary_cols],
}, index=binary_cols)
print(frac)
print("\nmin of continuous columns (should be >= 0):")
print(df_train[continuous_cols].min())

            fractional_in_train  originally_missing_in_train
HLA-B27                    1555                         1555
ANA                        2995                         3024
Anti-Ro                    2317                         2317
Anti-La                    2390                         2390
Anti-dsDNA                 3757                         3757
Anti-Sm                    4171                         4171

min of continuous columns (should be >= 0):
ESR          0.0
CRP          0.0
RF           0.0
Anti-CCP     0.0
C3          50.0
C4           5.0
dtype: float64


## 7. Engineered features

**What this does:** builds three extra features from existing columns. All are *row-wise* formulas (no learning from other rows), so they are leakage-free.

| Feature | Formula | Rationale / honest label |
|---|---|---|
| `Inflammation_Score` | ESR + CRP | Ad-hoc composite of two inflammation markers. **Not** a validated clinical score — call it an "ESR–CRP composite" in the paper. |
| `C3_C4_Ratio` | C3 / C4 | Hypothesis-driven: complement consumption differs in SLE. Guarded so C4 = 0 gives `NaN`, not `inf`. |
| `Autoantibody_Count` | sum of the 6 binary markers | Because markers are kept as continuous [0,1] estimates, this is a *sum of probabilities*, not a count of hard positives. Exploratory aggregate, not a diagnostic index. |

**Checks:** no `NaN`, no `inf`, all `float64`; and the per-disease means should differ in a sensible way (e.g. `Inflammation_Score` is lowest for `Normal`).

In [9]:
for d in (df_train, df_test):
    d['Inflammation_Score'] = d['ESR'] + d['CRP']
    d['C3_C4_Ratio'] = d['C3'] / d['C4'].replace(0, np.nan)   # np.nan keeps float dtype (pd.NA would make it object)
    d['Autoantibody_Count'] = d[binary_cols].sum(axis=1)

eng = ['Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']
for d, name in ((df_train, 'train'), (df_test, 'test')):
    assert d[eng].dtypes.eq('float64').all(), f"{name}: non-float engineered column"
    assert np.isfinite(d[eng].to_numpy()).all(), f"{name}: NaN/inf in engineered features"
print("Engineered features OK (float64, finite).")
df_train.groupby('Disease')[eng].mean().round(2)

Engineered features OK (float64, finite).


,Inflammation_Score,C3_C4_Ratio,Autoantibody_Count
Disease,,,
Ankylosing Spondylitis,53.39,3.96,3.44
Normal,10.92,4.09,3.25
Psoriatic Arthritis,62.11,3.98,2.97
Reactive Arthritis,40.48,4.02,3.53
Rheumatoid Arthritis,55.40,4.07,3.07
Sjögren's Syndrome,10.51,4.01,4.38
Systemic Lupus Erythematosus,10.26,10.00,4.38


## 8. Encode the target labels

**What this does:** `LabelEncoder` turns the 7 disease names into integers 0–6 (alphabetical). Fitted on the training labels; the test labels are only transformed.

**Why keep the mapping:** predictions come out as numbers; we need the mapping to translate them back to disease names in reports and SHAP plots. It is also saved to disk in section 12.

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_train['Disease_encoded'] = le.fit_transform(df_train['Disease'])
df_test['Disease_encoded']  = le.transform(df_test['Disease'])

label_mapping = {name: int(i) for i, name in enumerate(le.classes_)}
label_mapping

{'Ankylosing Spondylitis': 0,
 'Normal': 1,
 'Psoriatic Arthritis': 2,
 'Reactive Arthritis': 3,
 'Rheumatoid Arthritis': 4,
 "Sjögren's Syndrome": 5,
 'Systemic Lupus Erythematosus': 6}

## 9. Define the two feature sets

**What this does:** builds the two tables the experiments compare.
* **raw** — the 14 original columns.
* **extended** — raw + the 2 missingness flags + the 3 engineered features = 19 columns.

**Note:** we tested three further features from the literature (Seronegative flag, ESR×CRP, RF×Anti-CCP). None improved fold-safe CV macro-F1 beyond noise (0.8261 baseline vs 0.8227–0.8251), so they are **not** added. That negative result is kept for the paper (`results/new_feature_ablation.csv`).

**Expected shapes:** `(9668, 14)` and `(9668, 19)` for train; `(2417, 14)` and `(2417, 19)` for test.

In [11]:
raw_features = ['Age', 'Gender', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27',
                'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4']
extended_features = raw_features + ['RF_was_missing', 'Anti-CCP_was_missing',
                                    'Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']

X_raw_train, X_raw_test = df_train[raw_features].copy(), df_test[raw_features].copy()
X_ext_train, X_ext_test = df_train[extended_features].copy(), df_test[extended_features].copy()
y_train, y_test = df_train['Disease_encoded'], df_test['Disease_encoded']

print("raw:", X_raw_train.shape, X_raw_test.shape)
print("ext:", X_ext_train.shape, X_ext_test.shape)
assert X_raw_train.shape == (9668, 14) and X_ext_train.shape == (9668, 19)
assert X_raw_test.shape == (2417, 14) and X_ext_test.shape == (2417, 19)

raw: (9668, 14) (2417, 14)
ext: (9668, 19) (2417, 19)


## 10. Scale continuous features (fit on train only)

**What this does:** `StandardScaler` rewrites each continuous column as (value − train mean) / train standard deviation. Binary columns and the two missingness flags are left alone.

**Why:** Logistic Regression (and later SVM/KNN) are sensitive to feature scale; tree models (RF, XGBoost) are not, but we keep one scaled table so every model sees the same inputs. The scaler is fit on train only, so the test set is transformed with *training* statistics.

**Expected:** train columns have mean ≈ 0 and std ≈ 1 (checked below); test columns will be *close to but not exactly* 0/1 — that is correct and proves the scaler was not fit on test.

In [12]:
from sklearn.preprocessing import StandardScaler

scale_raw = continuous_cols + ['Age']
scale_ext = continuous_cols + ['Age', 'Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']

scaler_raw = StandardScaler()
X_raw_train_scaled, X_raw_test_scaled = X_raw_train.copy(), X_raw_test.copy()
X_raw_train_scaled[scale_raw] = scaler_raw.fit_transform(X_raw_train[scale_raw])
X_raw_test_scaled[scale_raw]  = scaler_raw.transform(X_raw_test[scale_raw])

scaler_ext = StandardScaler()
X_ext_train_scaled, X_ext_test_scaled = X_ext_train.copy(), X_ext_test.copy()
X_ext_train_scaled[scale_ext] = scaler_ext.fit_transform(X_ext_train[scale_ext])
X_ext_test_scaled[scale_ext]  = scaler_ext.transform(X_ext_test[scale_ext])

assert np.allclose(X_ext_train_scaled[scale_ext].mean(), 0, atol=1e-8)
assert np.allclose(X_ext_train_scaled[scale_ext].std(ddof=0), 1, atol=1e-8)
print("train (ext) mean/std OK. Test-set means (small but non-zero is expected):")
print(X_ext_test_scaled[scale_ext].mean().round(3))

train (ext) mean/std OK. Test-set means (small but non-zero is expected):
ESR                  -0.004
CRP                   0.016
RF                   -0.023
Anti-CCP             -0.021
C3                    0.029
C4                    0.005
Age                   0.001
Inflammation_Score    0.004
C3_C4_Ratio           0.023
Autoantibody_Count   -0.030
dtype: float64


## 11. Cross-check: does the shared CV code reproduce this notebook exactly?

**What this does:** calls `preprocess_fold` from `src/preprocessing/cv_pipeline.py` — the exact function the cross-validation notebooks use — with the un-imputed train/test copies from section 4 (treating "train" as the fold's training part and "test" as its validation part), then compares every output table against what this notebook just built.

**Why:** this notebook (used for the single train/test split) and the CV function (used per fold) implement the same steps in two places. If they ever differ, single-split and CV results would not be comparable. A pass here means: same imputation, same clipping, same engineered features, same scaling, same label encoding.

**Expected:** `ALL CHECKS PASSED`. If a check fails, the message names the table and the maximum difference — inspect that output.

In [13]:
from src.preprocessing.cv_pipeline import preprocess_fold

Xr_tr, Xr_va, Xe_tr, Xe_va, y_tr, y_va = preprocess_fold(df_train_raw, df_test_raw)

pairs = {
    'X_raw_train': (X_raw_train_scaled, Xr_tr), 'X_raw_test': (X_raw_test_scaled, Xr_va),
    'X_ext_train': (X_ext_train_scaled, Xe_tr), 'X_ext_test': (X_ext_test_scaled, Xe_va),
}
for name, (mine, ref) in pairs.items():
    assert list(mine.columns) == list(ref.columns), f"{name}: column mismatch"
    assert mine.index.equals(ref.index), f"{name}: row order mismatch"
    a, b = mine.to_numpy(dtype=float), ref.to_numpy(dtype=float)
    assert np.allclose(a, b, atol=1e-9), f"{name}: max abs diff {np.abs(a-b).max():.3e}"
assert (y_train.to_numpy() == y_tr.to_numpy()).all() and (y_test.to_numpy() == y_va.to_numpy()).all()
print("ALL CHECKS PASSED - notebook output == cv_pipeline.preprocess_fold output")

ALL CHECKS PASSED - notebook output == cv_pipeline.preprocess_fold output


## 12. Save the processed files

**What this does:** writes the scaled tables, labels, and the label mapping to `../data/processed/` (git-ignored). Downstream notebooks read these files instead of repeating the preprocessing.

**Reminder:** these files were produced with imputation/scaling fit on the *whole training set*. They are correct for single train/test-split experiments, but **cross-validation must not reuse them** — CV notebooks rebuild preprocessing inside every fold via `preprocess_fold`.

In [14]:
os.makedirs('../data/processed', exist_ok=True)

X_raw_train_scaled.to_csv('../data/processed/X_raw_train.csv', index=True)
X_raw_test_scaled.to_csv('../data/processed/X_raw_test.csv',  index=True)
X_ext_train_scaled.to_csv('../data/processed/X_ext_train.csv', index=True)
X_ext_test_scaled.to_csv('../data/processed/X_ext_test.csv',  index=True)
y_train.to_csv('../data/processed/y_train.csv', index=True)
y_test.to_csv('../data/processed/y_test.csv',  index=True)
with open('../data/processed/label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=2)

print("Saved:", sorted(os.listdir('../data/processed')))

Saved: ['X_ext_test.csv', 'X_ext_train.csv', 'X_raw_test.csv', 'X_raw_train.csv', 'label_mapping.json', 'y_test.csv', 'y_train.csv']


## Summary — what downstream notebooks receive

* `X_raw_*` (14 features) and `X_ext_*` (19 features), scaled with train-only statistics, for a stratified 80/20 split (9,668 / 2,417).
* `y_*` integer labels plus `label_mapping.json`.
* Verified: no NaN/inf, converged imputation, notebook output identical to `cv_pipeline.preprocess_fold`.
* The test set has **not** been used for any decision here.